## Preprocessing Congressional District CVAP

In [1]:
import pandas as pd

### Arkansas Congressional District CVAP Data Path

In [2]:
path = '../../data/Arkansas/ar-district-CVAP.csv'

### csv to pandas

In [3]:
cd_df = pd.read_csv(path)
cd_df

,geoname,lntitle,geoid,lnnumber,tot_est,tot_moe,adu_est,adu_moe,cit_est,cit_moe,cvap_est,cvap_moe
0,"Congressional District 1 (119th Congress), Ala...",Total,5001900US0101,1,735406,2960,570378,2085,723008,3245,559435,2209
1,"Congressional District 1 (119th Congress), Ala...",Not Hispanic or Latino,5001900US0101,2,697878,2900,546355,2112,692257,3013,541630,2133
2,"Congressional District 1 (119th Congress), Ala...",American Indian or Alaska Native Alone,5001900US0101,3,3322,347,2704,259,3315,348,2697,260
3,"Congressional District 1 (119th Congress), Ala...",Asian Alone,5001900US0101,4,9402,750,7201,532,7742,631,5603,446
4,"Congressional District 1 (119th Congress), Ala...",Black or African American Alone,5001900US0101,5,117993,2874,87763,1711,117006,2930,87067,1735
...,...,...,...,...,...,...,...,...,...,...,...,...
5715,Resident Commissioner District (at Large) (119...,Asian and White,5001900US7298,9,846,547,734,491,809,537,694,479
5716,Resident Commissioner District (at Large) (119...,Black or African American and White,5001900US7298,10,572,250,393,213,572,250,394,213
5717,Resident Commissioner District (at Large) (119...,American Indian or Alaska Native and Black or ...,5001900US7298,11,0,36,2,4,3,6,1,3
5718,Resident Commissioner District (at Large) (119...,Remainder of Two or More Race Responses,5001900US7298,12,1327,452,1217,412,1313,454,1197,414


### Isolate Columns

In [4]:
cd_df = cd_df[['geoname', 'lntitle', 'cvap_est']]
cd_df

,geoname,lntitle,cvap_est
0,"Congressional District 1 (119th Congress), Ala...",Total,559435
1,"Congressional District 1 (119th Congress), Ala...",Not Hispanic or Latino,541630
2,"Congressional District 1 (119th Congress), Ala...",American Indian or Alaska Native Alone,2697
3,"Congressional District 1 (119th Congress), Ala...",Asian Alone,5603
4,"Congressional District 1 (119th Congress), Ala...",Black or African American Alone,87067
...,...,...,...
5715,Resident Commissioner District (at Large) (119...,Asian and White,694
5716,Resident Commissioner District (at Large) (119...,Black or African American and White,394
5717,Resident Commissioner District (at Large) (119...,American Indian or Alaska Native and Black or ...,1
5718,Resident Commissioner District (at Large) (119...,Remainder of Two or More Race Responses,1197


### Cleaning and Renaming

In [5]:
cd_df = cd_df.rename(columns={'geoname':'District', 'lntitle':'Group', 'cvap_est':'Population'})
cd_df

,District,Group,Population
0,"Congressional District 1 (119th Congress), Ala...",Total,559435
1,"Congressional District 1 (119th Congress), Ala...",Not Hispanic or Latino,541630
2,"Congressional District 1 (119th Congress), Ala...",American Indian or Alaska Native Alone,2697
3,"Congressional District 1 (119th Congress), Ala...",Asian Alone,5603
4,"Congressional District 1 (119th Congress), Ala...",Black or African American Alone,87067
...,...,...,...
5715,Resident Commissioner District (at Large) (119...,Asian and White,694
5716,Resident Commissioner District (at Large) (119...,Black or African American and White,394
5717,Resident Commissioner District (at Large) (119...,American Indian or Alaska Native and Black or ...,1
5718,Resident Commissioner District (at Large) (119...,Remainder of Two or More Race Responses,1197


In [6]:
districts = [
    "Congressional District 1 (119th Congress), Arkansas",
    "Congressional District 2 (119th Congress), Arkansas",
    "Congressional District 3 (119th Congress), Arkansas",
    "Congressional District 4 (119th Congress), Arkansas"
]

cd_df = cd_df[cd_df['District'].isin(districts)]
cd_df

,District,Group,Population
221,"Congressional District 1 (119th Congress), Ark...",Total,568995
222,"Congressional District 1 (119th Congress), Ark...",Not Hispanic or Latino,555095
223,"Congressional District 1 (119th Congress), Ark...",American Indian or Alaska Native Alone,1366
224,"Congressional District 1 (119th Congress), Ark...",Asian Alone,2310
225,"Congressional District 1 (119th Congress), Ark...",Black or African American Alone,90719
226,"Congressional District 1 (119th Congress), Ark...",Native Hawaiian or Other Pacific Islander Alone,298
227,"Congressional District 1 (119th Congress), Ark...",White Alone,446068
228,"Congressional District 1 (119th Congress), Ark...",American Indian or Alaska Native and White,8432
229,"Congressional District 1 (119th Congress), Ark...",Asian and White,940
230,"Congressional District 1 (119th Congress), Ark...",Black or African American and White,3468


In [7]:
relevant_groups = ['Total', 'Black or African American Alone', 'White Alone', 'Hispanic or Latino']
cd_df = cd_df[cd_df['Group'].isin(relevant_groups)]
cd_df

,District,Group,Population
221,"Congressional District 1 (119th Congress), Ark...",Total,568995
225,"Congressional District 1 (119th Congress), Ark...",Black or African American Alone,90719
227,"Congressional District 1 (119th Congress), Ark...",White Alone,446068
233,"Congressional District 1 (119th Congress), Ark...",Hispanic or Latino,13902
234,"Congressional District 2 (119th Congress), Ark...",Total,569348
238,"Congressional District 2 (119th Congress), Ark...",Black or African American Alone,114846
240,"Congressional District 2 (119th Congress), Ark...",White Alone,414225
246,"Congressional District 2 (119th Congress), Ark...",Hispanic or Latino,19625
247,"Congressional District 3 (119th Congress), Ark...",Total,543233
251,"Congressional District 3 (119th Congress), Ark...",Black or African American Alone,14522


In [8]:
grouped = cd_df[cd_df['Group'] != 'Total'].groupby('District')['Population'].sum().reset_index()
totals = cd_df[cd_df['Group'] == 'Total'][['District', 'Population']]

merged = totals.merge(grouped, on='District', suffixes=('_Total', '_Known'))
merged['Other'] = merged['Population_Total'] - merged['Population_Known']

other_rows = merged[['District', 'Other']].rename(columns={'Other': 'Population'})
other_rows['Group'] = 'Other'

cd_df = pd.concat([cd_df, other_rows[['District', 'Group', 'Population']]], ignore_index=True)

cd_df

,District,Group,Population
0,"Congressional District 1 (119th Congress), Ark...",Total,568995
1,"Congressional District 1 (119th Congress), Ark...",Black or African American Alone,90719
2,"Congressional District 1 (119th Congress), Ark...",White Alone,446068
3,"Congressional District 1 (119th Congress), Ark...",Hispanic or Latino,13902
4,"Congressional District 2 (119th Congress), Ark...",Total,569348
5,"Congressional District 2 (119th Congress), Ark...",Black or African American Alone,114846
6,"Congressional District 2 (119th Congress), Ark...",White Alone,414225
7,"Congressional District 2 (119th Congress), Ark...",Hispanic or Latino,19625
8,"Congressional District 3 (119th Congress), Ark...",Total,543233
9,"Congressional District 3 (119th Congress), Ark...",Black or African American Alone,14522


In [9]:
cd_df = cd_df.sort_values(by='District')
cd_df = cd_df.reset_index()

In [10]:
cd_df.drop(columns='index', inplace=True)
cd_df

,District,Group,Population
0,"Congressional District 1 (119th Congress), Ark...",Total,568995
1,"Congressional District 1 (119th Congress), Ark...",Black or African American Alone,90719
2,"Congressional District 1 (119th Congress), Ark...",White Alone,446068
3,"Congressional District 1 (119th Congress), Ark...",Hispanic or Latino,13902
4,"Congressional District 1 (119th Congress), Ark...",Other,18306
5,"Congressional District 2 (119th Congress), Ark...",Total,569348
6,"Congressional District 2 (119th Congress), Ark...",Black or African American Alone,114846
7,"Congressional District 2 (119th Congress), Ark...",White Alone,414225
8,"Congressional District 2 (119th Congress), Ark...",Hispanic or Latino,19625
9,"Congressional District 2 (119th Congress), Ark...",Other,20652


### Creating New DataFrame

In [11]:
unique_districts = cd_df['District'].unique()
unique_districts

<StringArray>
['Congressional District 1 (119th Congress), Arkansas',
 'Congressional District 2 (119th Congress), Arkansas',
 'Congressional District 3 (119th Congress), Arkansas',
 'Congressional District 4 (119th Congress), Arkansas']
Length: 4, dtype: str

In [12]:
summary = []
district_num = 1

for district in unique_districts:
    data = {}

    data['State'] = 'Arkansas'
    data['District'] = district_num
    data['Total CVAP'] = cd_df[(cd_df['District'] == district) & (cd_df['Group'] == 'Total')]['Population'].values[0]
    data['White CVAP'] = cd_df[(cd_df['District'] == district) & (cd_df['Group'] == 'White Alone')]['Population'].values[0]
    data['Black CVAP'] = cd_df[(cd_df['District'] == district) & (cd_df['Group'] == 'Black or African American Alone')]['Population'].values[0]
    data['Latino CVAP'] = cd_df[(cd_df['District'] == district) & (cd_df['Group'] == 'Hispanic or Latino')]['Population'].values[0]
    data['Other CVAP'] = cd_df[(cd_df['District'] == district) & (cd_df['Group'] == 'Other')]['Population'].values[0]
    district_num += 1

    summary.append(data)

In [13]:
district_cvap = pd.DataFrame(summary)
district_cvap

,State,District,Total CVAP,White CVAP,Black CVAP,Latino CVAP,Other CVAP
0,Arkansas,1,568995,446068,90719,13902,18306
1,Arkansas,2,569348,414225,114846,19625,20652
2,Arkansas,3,543233,436910,14522,52980,38821
3,Arkansas,4,564359,411979,110143,23680,18557


### Export New DataFrame

In [15]:
district_cvap.to_csv('../output/Arkansas/ar-district-CVAP.csv')